# Self-Attention from Scratch: NumPy and PyTorch

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Where:
- $Q$ = Query matrix
- $K$ = Key matrix
- $V$ = Value matrix
- $d_k$ = Dimension of the key vectors

This notebook demonstrates how to implement the self-attention mechanism from scratch using both NumPy and PyTorch. We'll start with a NumPy implementation for conceptual clarity, then move to a PyTorch version suitable for deep learning workflows.

Each code cell is preceded by a markdown cell explaining its purpose and the logic behind the code.

# Self-Attention from Scratch: NumPy and PyTorch

This notebook demonstrates how to implement the self-attention mechanism from scratch using both NumPy and PyTorch. We'll start with a NumPy implementation for conceptual clarity, then move to a PyTorch version suitable for deep learning workflows.

Each code cell is preceded by a markdown cell explaining its purpose and the logic behind the code.

In [1]:
import numpy as np

class SimpleSelfAttentionNumpy:
    def __init__(self, embedding_dim):
        self.embedding_dim = embedding_dim
        # Initialize weights and biases for Q, K, V projections
        self.query_weight = np.random.randn(embedding_dim, embedding_dim) * 0.01
        self.query_bias = np.zeros(embedding_dim)
        self.key_weight = np.random.randn(embedding_dim, embedding_dim) * 0.01
        self.key_bias = np.zeros(embedding_dim)
        self.value_weight = np.random.randn(embedding_dim, embedding_dim) * 0.01
        self.value_bias = np.zeros(embedding_dim)

    def linear(self, x, weight, bias):
        # x: (batch_size, seq_len, embedding_dim)
        return np.dot(x, weight) + bias  # bias is broadcasted

    def softmax(self, x, axis=-1):
        # Numerically stable softmax
        x_max = np.max(x, axis=axis, keepdims=True)
        e_x = np.exp(x - x_max)
        return e_x / e_x.sum(axis=axis, keepdims=True)

    def forward(self, x):
        """
        x: (batch_size, seq_len, embedding_dim)
        returns: (batch_size, seq_len, embedding_dim)
        """
        # 1. Linear projections
        Q = self.linear(x, self.query_weight, self.query_bias)
        K = self.linear(x, self.key_weight, self.key_bias)
        V = self.linear(x, self.value_weight, self.value_bias)

        # 2. Compute attention scores
        # (batch_size, seq_len, embed_dim) @ (batch_size, embed_dim, seq_len)
        attention_scores = np.matmul(Q, K.transpose(0, 2, 1))
        # 3. Scale by sqrt(embedding_dim)
        attention_scores /= np.sqrt(self.embedding_dim)
        # 4. Softmax to get attention weights
        attention_weights = self.softmax(attention_scores, axis=-1)
        # 5. Weighted sum of values
        output = np.matmul(attention_weights, V)
        return output

## NumPy Implementation: Self-Attention Layer

This cell defines a simple self-attention layer using NumPy. The class includes:
- Initialization of weights and biases for the Query, Key, and Value projections.
- Linear transformation and softmax helper functions.
- The `forward` method, which computes the self-attention output for a batch of sequences.

**Key variable names:**
- `query_weight`, `key_weight`, `value_weight`: Weight matrices for projections.
- `query_bias`, `key_bias`, `value_bias`: Bias vectors for projections.
- `linear`: Applies a linear transformation.
- `softmax`: Numerically stable softmax function.
- `forward`: Runs the self-attention computation.

In [6]:
# Set parameters for the example
batch_size = 2
sequence_length = 4
embedding_dim = 3

# Create random input (batch_size, sequence_length, embedding_dim)
input_data = np.random.randn(batch_size, sequence_length, embedding_dim)

# Create the self-attention layer
attention_layer = SimpleSelfAttentionNumpy(embedding_dim)

# Forward pass
output = attention_layer.forward(input_data)

# Print results
print("Input shape:", input_data.shape)
print("Output shape:", output.shape)
print("Output:\n", output)

Input shape: (2, 4, 3)
Output shape: (2, 4, 3)
Output:
 [[[ 0.00066445 -0.01176653 -0.00150804]
  [ 0.00066429 -0.01176804 -0.00150746]
  [ 0.0006643  -0.01176519 -0.00150918]
  [ 0.00066219 -0.01177117 -0.00150998]]

 [[ 0.00118866  0.01170225  0.00511953]
  [ 0.00119     0.01169925  0.00512283]
  [ 0.00119252  0.01169904  0.0051265 ]
  [ 0.00119208  0.01169826  0.00512706]]]
[[[ 0.00066445 -0.01176653 -0.00150804]
  [ 0.00066429 -0.01176804 -0.00150746]
  [ 0.0006643  -0.01176519 -0.00150918]
  [ 0.00066219 -0.01177117 -0.00150998]]

 [[ 0.00118866  0.01170225  0.00511953]
  [ 0.00119     0.01169925  0.00512283]
  [ 0.00119252  0.01169904  0.0051265 ]
  [ 0.00119208  0.01169826  0.00512706]]]


## Example: Running the NumPy Self-Attention Layer

This cell demonstrates how to use the `SimpleSelfAttentionNumpy` class. We:
- Create a random input tensor with shape `(batch_size, sequence_length, embedding_dim)`.
- Instantiate the self-attention layer.
- Run a forward pass and print the input and output shapes.

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleSelfAttentionTorch(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.embedding_dim = embedding_dim
        # Linear layers for Q, K, V projections
        self.query_proj = nn.Linear(embedding_dim, embedding_dim)
        self.key_proj = nn.Linear(embedding_dim, embedding_dim)
        self.value_proj = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        # x: (batch_size, sequence_length, embedding_dim)
        queries = self.query_proj(x)
        keys = self.key_proj(x)
        values = self.value_proj(x)
        # Compute attention scores (Q @ K^T)
        attention_scores = torch.matmul(queries, keys.transpose(-2, -1))
        # Scale by sqrt(embedding_dim)
        attention_scores = attention_scores / (self.embedding_dim ** 0.5)
        # Softmax to get attention weights
        attention_weights = F.softmax(attention_scores, dim=-1)
        # Weighted sum of values
        output = torch.matmul(attention_weights, values)
        return output

## PyTorch Implementation: Self-Attention Layer

This cell defines a self-attention layer using PyTorch's `nn.Module`. The implementation closely follows the NumPy version, but leverages PyTorch's built-in layers and tensor operations for deep learning workflows.

**Key variable names:**
- `query_proj`, `key_proj`, `value_proj`: Linear layers for projections.
- `forward`: Runs the self-attention computation.

In [5]:
# Set parameters for the example
batch_size = 2
sequence_length = 5
embedding_dim = 128

# Create a random input tensor
input_tensor = torch.randn(batch_size, sequence_length, embedding_dim)

# Initialize the self-attention model
self_attention_model = SimpleSelfAttentionTorch(embedding_dim)

# Pass the input through the model
output_tensor = self_attention_model(input_tensor)

print(f"Input shape: {input_tensor.shape}")
print(f"Output shape: {output_tensor.shape}")

Input shape: torch.Size([2, 5, 128])
Output shape: torch.Size([2, 5, 128])


## Example: Running the PyTorch Self-Attention Layer

This cell demonstrates how to use the `SimpleSelfAttentionTorch` class. We:
- Create a random input tensor with shape `(batch_size, sequence_length, embedding_dim)`.
- Instantiate the self-attention layer.
- Run a forward pass and print the input and output shapes.